# 🚀 STGCN + EVT-GPD + XLinear (KnowAir-V2 BTHSA)
Chạy trực tiếp trên Google Colab. Chọn Runtime -> T4 GPU trước.

In [ ]:
!pip install xarray netcdf4 pandas numpy torch scipy tqdm

### 1. Tải Dataset KnowAir-V2 
Tải trực tiếp từ Zenodo xuống Colab siêu nhanh.

In [ ]:
import os
os.makedirs('data', exist_ok=True)
!wget -nc -O data/dataset_bthsa.nc https://zenodo.org/records/15614907/files/dataset_bthsa.nc
!wget -nc -O data/stations_bthsa.csv https://zenodo.org/records/15614907/files/stations_bthsa.csv


### 2. Tiền xử lý & Adjacency Matrix

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

DATA_DIR = "data"
OUT_DIR = "data/clean"
OUT_DIR = "data/clean"

def haversine_dist(lat1, lon1, lat2, lon2):
    """Calculate the great circle distance between two points on the earth (specified in decimal degrees)"""
    # convert decimal degrees to radians 
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # haversine formula 
    dlat = lat2 - lat1 
    dlon = lon2 - lon1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    r = 6371 # Radius of earth in kilometers
    return c * r

def process_region(region_name):
    print(f"\nProcessing {region_name.upper()} region...")
    os.makedirs(OUT_DIR, exist_ok=True)
    
    # 1. Load Dataset
    nc_path = os.path.join(DATA_DIR, f"dataset_{region_name}.nc")
    csv_path = os.path.join(DATA_DIR, f"stations_{region_name}.csv")
    
    ds = xr.open_dataset(nc_path)
    stations_df = pd.read_csv(csv_path)
    
    num_stations = len(stations_df)
    
    # Target feature first, then meteorology
    feature_vars = ["PM2.5", "O3", "t2m", "d2m", "sp", "tp", "blh", "msdwswrf", "u100", "v100"]
    num_features = len(feature_vars)
    
    time_len = ds.dims['time']
    print(f"Data shape will be: (time={time_len}, stations={num_stations}, features={num_features})")
    
    # Create final numpy array
    # Memory efficient allocation
    out_array = np.zeros((time_len, num_stations, num_features), dtype=np.float32)
    
    print("Extracting features from NetCDF...")
    for f_idx, var_name in enumerate(tqdm(feature_vars)):
        # Data in NC is (time, station)
        val = ds[var_name].values
        # Assign to our out_array
        out_array[:, :, f_idx] = val
        
    out_npy = os.path.join(OUT_DIR, f"knowair_{region_name}.npy")
    print(f"Saving data array to {out_npy}...")
    np.save(out_npy, out_array)
    
    # 2. Build Adjacency Matrix
    print("Building adjacency matrix...")
    lats = stations_df['lat'].values
    lons = stations_df['lon'].values
    
    dist_mat = np.zeros((num_stations, num_stations))
    for i in range(num_stations):
        for j in range(i+1, num_stations):
            d = haversine_dist(lats[i], lons[i], lats[j], lons[j])
            dist_mat[i, j] = d
            dist_mat[j, i] = d
            
    # Gaussian kernel thresholding as in STGCN
    sigma2 = 2500.0
    epsilon = 0.1
    
    W = np.zeros((num_stations, num_stations))
    for i in range(num_stations):
        for j in range(num_stations):
            if i != j:
                w = np.exp(-(dist_mat[i, j] ** 2) / sigma2)
                if w >= epsilon:
                    W[i, j] = w
                    
    # Normalized Laplacian
    A = W + np.eye(num_stations)
    D = np.diag(np.power(A.sum(1), -0.5))
    D[np.isinf(D)] = 0.
    adj = D @ A @ D
    
    out_adj = os.path.join(OUT_DIR, f"adj_mat_knowair_{region_name}.npy")
    print(f"Saving adjacency matrix to {out_adj}...")
    np.save(out_adj, adj)
    
    print(f"Done processing {region_name}!")

if __name__ == "__main__":
    process_region("bthsa")  # Beijing-Tianjin-Hebei
    # process_region("yrd")  # Yangtze River Delta (optional, not enabled now to save RAM)



### 3. Khởi tạo Model (STGCN_XLinear_GPD)

In [ ]:
"""
Hybrid STGCN + XLinear + EVT-GPD Model v2 (PyTorch).

Thiết kế lại: KHÔNG thay thế Conv1D bằng Gating, mà KẾT HỢP cả hai.
  - Conv1D TimeBlock (GLU): bắt pattern thời gian CỤC BỘ (3 giờ liên tiếp)
  - XLinear Gating: lọc feature TOÀN CỤC (giữ tín hiệu quan trọng, bỏ noise)
  → Conv1D cho locality + Gating cho selectivity = best of both worlds

Kiến trúc mỗi block:
  TimeBlock(Conv1D+GLU) → XLinear Gating → Graph Conv → TimeBlock → Gating → BN
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


# ========================================================================
# XLinear Gating Block
# ========================================================================

class GatingBlock(nn.Module):
    """x * sigmoid(MLP(x)) — chọn lọc features quan trọng."""

    def __init__(self, d_model, hidden_dim, dropout=0.):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.gate(x)


# ========================================================================
# Enhanced TimeBlock: Conv1D GLU + XLinear Gating
# ========================================================================

class EnhancedTimeBlock(nn.Module):
    """
    Conv1D (GLU) + XLinear Gating.
    Conv1D bắt pattern cục bộ → Gating lọc noise toàn cục.
    """

    def __init__(self, in_channels, out_channels, kernel_size=3, gating_ff=64):
        super().__init__()
        # Conv1D + GLU (giữ nguyên từ STGCN)
        self.conv1 = nn.Conv2d(in_channels, out_channels, (1, kernel_size))
        self.conv2 = nn.Conv2d(in_channels, out_channels, (1, kernel_size))
        self.conv3 = nn.Conv2d(in_channels, out_channels, (1, kernel_size))

        # XLinear Gating (thêm mới): lọc theo chiều channel
        self.gating = GatingBlock(out_channels, gating_ff)

    def forward(self, X):
        """X: (batch, in_channels, nodes, seq_len)"""
        # Conv1D + GLU (local temporal patterns)
        v1 = self.conv1(X)
        v2 = torch.sigmoid(self.conv2(X))
        residual = self.conv3(X)
        out = residual + v1 * v2  # (batch, out_ch, nodes, seq-2)

        # XLinear Gating (global feature selection trên channel dimension)
        # Permute: (batch, ch, nodes, seq) → (batch, nodes, seq, ch)
        out = out.permute(0, 2, 3, 1)
        out = self.gating(out)
        # Permute back: (batch, nodes, seq, ch) → (batch, ch, nodes, seq)
        out = out.permute(0, 3, 1, 2)

        return out


# ========================================================================
# STX Block (Spatio-Temporal-XLinear Block)
# ========================================================================

class STXBlock(nn.Module):
    """
    Enhanced TimeBlock → Graph Conv → Enhanced TimeBlock → BatchNorm

    So với STGCN: thêm XLinear Gating sau mỗi TimeBlock.
    So với XLinear thuần: giữ Conv1D cho locality.
    """

    def __init__(self, in_channels, spatial_channels, out_channels, num_nodes, gating_ff=64):
        super().__init__()
        self.temporal1 = EnhancedTimeBlock(in_channels, out_channels, gating_ff=gating_ff)
        self.theta = nn.Parameter(torch.FloatTensor(out_channels, spatial_channels))
        nn.init.xavier_uniform_(self.theta)
        self.temporal2 = EnhancedTimeBlock(spatial_channels, out_channels, gating_ff=gating_ff)
        self.batch_norm = nn.BatchNorm2d(num_nodes)

    def forward(self, X, A_hat):
        """X: (batch, in_ch, nodes, seq_len), A_hat: (nodes, nodes)"""
        # Temporal + Gating 1
        t = self.temporal1(X)

        # Spectral Graph Conv
        t_p = t.permute(0, 3, 2, 1)  # (batch, seq, nodes, ch)
        s = torch.einsum('btnd,nm->btmd', t_p, A_hat)
        s = torch.relu(torch.matmul(s, self.theta))
        s = s.permute(0, 3, 2, 1)  # (batch, spatial_ch, nodes, seq)

        # Temporal + Gating 2
        t2 = self.temporal2(s)

        # BatchNorm
        out = self.batch_norm(t2.permute(0, 2, 1, 3))
        return out.permute(0, 2, 1, 3)


# ========================================================================
# Cross-Variable Attention (từ XLinear)
# ========================================================================

class CrossVariableGating(nn.Module):
    """
    Lọc feature nào quan trọng cho prediction (từ XLinear Forcast_with_exogenous).
    Áp dụng Gating trên chiều channels sau khi qua STX blocks.
    """

    def __init__(self, num_channels, hidden_dim):
        super().__init__()
        self.gate = GatingBlock(num_channels, hidden_dim)

    def forward(self, X):
        """X: (batch, channels, nodes, seq_left)"""
        # Permute: (batch, ch, nodes, seq) → (batch, nodes, seq, ch)
        X = X.permute(0, 2, 3, 1)
        X = self.gate(X)
        # (batch, nodes, seq, ch) → (batch, ch, nodes, seq)
        return X.permute(0, 3, 1, 2)


# ========================================================================
# Main Model
# ========================================================================

class STGCN_XLinear(nn.Module):
    """
    STGCN + XLinear Hybrid v2.

    Kiến trúc:
      2x STXBlock (Conv1D+Gating → GCN → Conv1D+Gating → BN)
      → Cross-Variable Gating
      → Last Enhanced TimeBlock
      → FC Head

    Ưu điểm kết hợp:
      - Conv1D (STGCN): bắt pattern cục bộ 3h liên tiếp (ngày/đêm, rush hour)
      - Gating (XLinear): lọc bỏ noise, tăng cường tín hiệu quan trọng
      - Cross-var Gating: tự động chọn features ảnh hưởng nhất đến target
      - Graph Conv: quan hệ không gian giữa trạm
    """

    def __init__(self, num_nodes, num_features, num_timesteps_input,
                 num_timesteps_output, gating_ff=64):
        super().__init__()

        # STXBlock 1: (num_features -> 16 spatial -> 64 out)
        self.block1 = STXBlock(num_features, 16, 64, num_nodes, gating_ff)
        # STXBlock 2: (64 in -> 16 spatial -> 64 out)
        self.block2 = STXBlock(64, 16, 64, num_nodes, gating_ff)

        self.cross_var = CrossVariableGating(64, gating_ff)
        self.last_temporal = EnhancedTimeBlock(64, 64, gating_ff=gating_ff)

        # Output head
        # STXBlock has 2 EnhancedTimeBlocks (kernel=3 without padding -> length decreases by 2 per block)
        # 1 STXBlock reduces length by 2 * 2 = 4
        # 2 STXBlocks reduce length by 8
        # last_temporal reduces length by 2
        # Total reduction = 10
        out_time_len = num_timesteps_input - 10
        self.fc1 = nn.Linear(out_time_len * 64, 256)
        self.fc2 = nn.Linear(256, num_timesteps_output)

    def forward(self, A_hat, X):
        """X: (batch, seq_len, nodes, features)"""
        X = X.permute(0, 3, 2, 1)  # → (batch, features, nodes, seq)

        out = self.block1(X, A_hat)
        out = self.block2(out, A_hat)

        # Cross-Variable Gating: lọc channels quan trọng
        out = self.cross_var(out)

        out = self.last_temporal(out)

        b, c, n, t = out.shape
        out = out.permute(0, 2, 1, 3).reshape(b, n, c * t)

        out = F.relu(self.fc1(out))
        out = self.fc2(out)

        return out  # (batch, nodes, pred_len)


# ========================================================================
# EVT-GPD Loss
# ========================================================================

class EVTGPDLoss(nn.Module):
    def __init__(self, xi, sigma, mean_vals, std_vals,
                 threshold=60.0, beta_1=0.99, beta_2=0.01):
        super().__init__()
        self.threshold, self.beta_1, self.beta_2 = threshold, beta_1, beta_2
        self.warmup_done = False
        self.register_buffer('xi', torch.FloatTensor(xi))
        self.register_buffer('sig', torch.FloatTensor(sigma))
        self.register_buffer('mean_val', torch.tensor(mean_vals, dtype=torch.float32))
        self.register_buffer('std_val', torch.tensor(std_vals, dtype=torch.float32))

    def set_warmup(self, enabled):
        self.warmup_done = enabled

    def forward(self, y_pred, y_true):
        mse = F.mse_loss(y_pred, y_true)
        if not self.warmup_done:
            return mse
        y_d = y_pred.detach() * self.std_val + self.mean_val
        xi = self.xi.unsqueeze(0).unsqueeze(-1)
        sig = self.sig.unsqueeze(0).unsqueeze(-1)
        z_safe = torch.clamp(1.0 + xi * y_d / (sig + 1e-6), min=1e-6)
        gpd = torch.log(sig + 1e-6) + (1 + 1 / (xi + 1e-6)) * torch.log(z_safe)
        mask = (y_d > self.threshold).float()
        penalty = torch.clamp((gpd * mask).mean(), -50, 50)
        return self.beta_1 * mse + self.beta_2 * penalty



### 4. Bắt đầu Train

In [ ]:
"""
Hybrid STGCN + XLinear + EVT-GPD — Training Script.
Pipeline: Load Data → Graph → POT Fitting → Train → Evaluate.
"""

import argparse
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import time
import os
import glob
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from torch.utils.data import Dataset, DataLoader
from scipy.stats import genpareto

# Model components are defined in the cell above


# ========================================================================
# DATA
# ========================================================================
class AQIDataset(Dataset):
    def __init__(self, data, seq_len, pre_len, target_idx):
        self.data, self.seq_len, self.pre_len, self.target_idx = data, seq_len, pre_len, target_idx
        self.n = len(data) - seq_len - pre_len + 1

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        X = self.data[i:i + self.seq_len, :, :]
        y = self.data[i + self.seq_len:i + self.seq_len + self.pre_len, :, self.target_idx]
        return torch.FloatTensor(X), torch.FloatTensor(y.T)


def load_data(data_npy_path, batch_size, seq_len, pre_len, target_idx, limit_years=None):
    print(f"Loading data from {data_npy_path}")
    # raw is [time, num_nodes, num_features]
    raw = np.load(data_npy_path)
    
    if limit_years is not None:
        # 1 year = 365 * 24 = 8760 hours
        max_len = limit_years * 8760
        if raw.shape[0] > max_len:
            raw = raw[-max_len:] # Take last N years to save memory
            print(f"Limited data to last {limit_years} years: shape {raw.shape}")

    ts = int(len(raw) * 0.7)
    vs = int(len(raw) * 0.1)

    mean = np.mean(raw[:ts], axis=(0, 1))
    std = np.std(raw[:ts], axis=(0, 1))
    normed = (raw - mean) / (std + 1e-5)

    loaders = [
        DataLoader(AQIDataset(normed[s:e], seq_len, pre_len, target_idx),
                   batch_size=batch_size, shuffle=(i == 0))
        for i, (s, e) in enumerate([(0, ts), (ts, ts + vs), (ts + vs, len(raw))])
    ]
    return loaders[0], loaders[1], loaders[2], mean, std


# ========================================================================
# GRAPH
# ========================================================================
def build_graph(adj_npy_path):
    print(f"Loading adjacency matrix from {adj_npy_path}")
    return np.load(adj_npy_path)


# ========================================================================
# POT FITTING
# ========================================================================
def fit_gpd(data_dir, threshold, target_col, num_stations):
    xi_list, sig_list = [], []
    for i in range(1, num_stations + 1):
        path = os.path.join(data_dir, f"station_{i}.csv")
        try:
            df = pd.read_csv(path)
            if target_col not in df.columns:
                print(f"  Station {i}: column '{target_col}' missing, using defaults")
                xi_list.append(0.1); sig_list.append(10.0)
                continue
            vals = df[target_col].dropna().values
            exc = vals[:int(len(vals) * 0.7)]
            exc = exc[exc > threshold] - threshold
            if len(exc) < 10:
                xi_list.append(0.1); sig_list.append(10.0)
            else:
                c, _, s = genpareto.fit(exc, floc=0)
                xi_list.append(c); sig_list.append(s)
                print(f"  Station {i}: xi={c:.4f}, sigma={s:.4f} ({len(exc)} exc.)")
        except Exception as e:
            print(f"  Station {i}: error ({e}), using defaults")
            xi_list.append(0.1); sig_list.append(10.0)
    return np.array(xi_list), np.array(sig_list)


# ========================================================================
# EVALUATE
# ========================================================================
def evaluate(model, loader, A_hat, criterion, device, return_preds=False):
    model.eval()
    total_loss, preds, targs = 0., [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(A_hat, X)
            total_loss += criterion(out, y).item() * X.size(0)
            if return_preds:
                preds.append(out.cpu().numpy())
                targs.append(y.cpu().numpy())
    avg = total_loss / len(loader.dataset)
    if return_preds:
        return avg, np.concatenate(preds), np.concatenate(targs)
    return avg


# ========================================================================
# TRAIN
# ========================================================================
def train(args):
    torch.manual_seed(42)
    np.random.seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(42)
        torch.backends.cudnn.deterministic = True

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    root = "data"
    data_npy = os.path.join(root, "clean", "knowair_bthsa.npy")
    adj_npy = os.path.join(root, "clean", "adj_mat_knowair_bthsa.npy")

    # Data
    print("\n[1/4] Loading data...")
    train_ld, val_ld, test_ld, mean, std = load_data(
        data_npy, args.batch_size, args.seq_len, args.pre_len, args.target_idx, args.limit_years
    )
    raw = np.load(data_npy)
    if args.limit_years is not None:
        max_len = args.limit_years * 8760
        if raw.shape[0] > max_len:
            raw = raw[-max_len:]
            
    nf = mean.shape[0]
    print(f"  Features: {nf}, Target idx: {args.target_idx}")
    print(f"  Train/Val/Test: {len(train_ld.dataset)}/{len(val_ld.dataset)}/{len(test_ld.dataset)}")

    # Graph
    print("\n[2/4] Building graph...")
    A_hat_np = build_graph(adj_npy)
    nn_nodes = A_hat_np.shape[0]
    A_hat = torch.FloatTensor(A_hat_np).to(device)
    print(f"  Nodes: {nn_nodes}")

    # POT
    print(f"\n[3/4] Fitting GPD (threshold={args.threshold})...")
    params = []
    for i in range(nn_nodes):
        valid = raw[:, i, args.target_idx]
        valid = valid[~np.isnan(valid)]
        exc = valid[valid > args.threshold] - args.threshold
        if len(exc) > 10:
            c, loc, scale = genpareto.fit(exc, floc=0)
            params.append(float(c))
            params.append(float(scale))
        else:
            params.append(0.1)
            params.append(10.0)
            
    # reshaping for tensor
    xi_arr = np.array(params[0::2], dtype=np.float32)
    sig_arr = np.array(params[1::2], dtype=np.float32)
    xi = torch.FloatTensor(xi_arr)
    sig = torch.FloatTensor(sig_arr)
    
    t_mean, t_std = float(mean[args.target_idx]), float(std[args.target_idx])

    evt_loss = EVTGPDLoss(
        xi=xi, sigma=sig, mean_vals=t_mean, std_vals=t_std,
        threshold=args.threshold, beta_1=args.beta1, beta_2=args.beta2
    ).to(device)
    mse_loss = nn.MSELoss()

    # Model
    print(f"\n[4/4] Building Hybrid STGCN + XLinear + EVT-GPD v2...")
    model = STGCN_XLinear(
        num_nodes=nn_nodes,
        num_features=nf,
        num_timesteps_input=args.seq_len,
        num_timesteps_output=args.pre_len,
        gating_ff=args.t_ff
    ).to(device)

    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {params:,}")
    print(f"  Gating FF: {args.t_ff}")
    print(f"  GPD Warmup: {args.warmup} epochs | beta1={args.beta1}, beta2={args.beta2}")

    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=7, factor=0.5, min_lr=1e-6)

    best_val, patience_cnt = float('inf'), 0
    os.makedirs("checkpoints", exist_ok=True)

    print(f"\n{'='*75}")
    print(f"{'Epoch':>6} | {'Train Loss':>12} | {'Val MSE':>10} | {'LR':>10} | {'Time':>6} | Status")
    print(f"{'='*75}")

    for epoch in range(args.epochs):
        t0 = time.time()
        model.train()
        use_evt = epoch >= args.warmup
        evt_loss.set_warmup(use_evt)
        label = "EVT" if use_evt else "MSE"

        ep_loss = 0.
        for X, y in train_ld:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(A_hat, X)
            loss = evt_loss(out, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            ep_loss += loss.item() * X.size(0)

        ep_loss /= len(train_ld.dataset)
        val = evaluate(model, val_ld, A_hat, mse_loss, device)
        scheduler.step(val)
        lr = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0

        if val < best_val:
            best_val = val
            patience_cnt = 0
            torch.save(model.state_dict(), 'checkpoints/best_stgcn_xlinear.pth')
            status = f"★ Best ({best_val:.4f})"
        else:
            patience_cnt += 1
            status = f"wait {patience_cnt}/{args.patience}"

        print(f"{epoch+1:>6} | {label:>4} {ep_loss:>7.4f} | {val:>10.4f} | {lr:>10.6f} | {elapsed:>5.1f}s | {status}")

        if patience_cnt >= args.patience:
            print(f"\n  Early stopping at epoch {epoch + 1}!")
            break

    # Test
    print(f"\n{'='*75}")
    print("TESTING — Hybrid STGCN + XLinear + EVT-GPD")
    print(f"{'='*75}")

    model.load_state_dict(torch.load('checkpoints/best_stgcn_xlinear.pth', weights_only=True))
    tl, pn, tn = evaluate(model, test_ld, A_hat, mse_loss, device, return_preds=True)

    pr = pn * t_std + t_mean
    tr = tn * t_std + t_mean
    pf, tf_ = pr.flatten(), tr.flatten()

    rmse = np.sqrt(mean_squared_error(tf_, pf))
    mae = mean_absolute_error(tf_, pf)
    r2 = r2_score(tf_, pf)

    print(f"  Test MSE (norm)  : {tl:.4f}")
    print(f"  Test RMSE ({target_name:>10}): {rmse:.2f}")
    print(f"  Test MAE  ({target_name:>10}): {mae:.2f}")
    print(f"  Test R²-Score    : {r2:.4f}")


class Args:
    pass
args = Args()
args.seq_len = 72
args.pre_len = 24
args.batch_size = 16
args.epochs = 100
args.lr = 0.001
args.target_idx = 0
args.limit_years = 2
args.threshold = 50.0
args.warmup = 15
args.patience = 20
args.beta1 = 0.99
args.beta2 = 0.01
args.d_model = 64
args.t_ff = 128
args.num_nodes = 228
args.in_dim = 10

import os
os.makedirs("data/clean", exist_ok=True)

print("Starting Colab Training...")
# train(args) # Uncomment to train
if False:
    parser = argparse.ArgumentParser(description='Hybrid STGCN + XLinear + EVT-GPD')
    parser.add_argument('--seq_len', type=int, default=72)
    parser.add_argument('--pre_len', type=int, default=24)
    parser.add_argument('--batch_size', type=int, default=16)
    parser.add_argument('--epochs', type=int, default=100)
    parser.add_argument('--lr', type=float, default=0.001)
    parser.add_argument('--target_idx', type=int, default=0, help='0 = PM2.5 in KnowAir')
    parser.add_argument('--limit_years', type=int, default=2, help='Only use last N years to save RAM')
    parser.add_argument('--threshold', type=float, default=50.0)
    parser.add_argument('--warmup', type=int, default=15)
    parser.add_argument('--patience', type=int, default=20)
    parser.add_argument('--beta1', type=float, default=0.99)
    parser.add_argument('--beta2', type=float, default=0.01)
    parser.add_argument('--d_model', type=int, default=64, help='XLinear temporal projection dim')
    parser.add_argument('--t_ff', type=int, default=128, help='Gating hidden dim')

    args = parser.parse_args()
    train(args)



train(args)